## Transformer Anatomy and Transformer Circuits

Run the cells in order with **Shift+Enter**. This notebook provides an executable path through the repository's cache and circuit experiments. Each experiment cell shows progress, a compact result summary, and the saved JSON path. The `.sh` files hold the configured examples; the Python files named below implement the analyses.

### 0. Notebook setup

Find the repository root and pass the active notebook interpreter to every script. The printed paths let you confirm which Python environment and GPU index will be used.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "requirements.txt").is_file():
    ROOT = ROOT.parent
if not (ROOT / "requirements.txt").is_file():
    raise RuntimeError("Open Demo/demo.ipynb from this repository")

ENV = os.environ.copy()
ENV["PYTHON_BIN"] = sys.executable
ENV.setdefault("GPU", "0")

def run_command(command, *, environment=None, result_dir=None):
    print("Running:", " ".join(str(part) for part in command), flush=True)
    process = subprocess.Popen(
        [str(part) for part in command], cwd=ROOT, env=environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}")
    print("Completed successfully.")
    if result_dir is not None:
        folder = ROOT / result_dir
        files = sorted(folder.rglob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True) if folder.exists() else []
        print(f"JSON results in {result_dir}: {len(files)}")
        if files:
            print("Latest result:", files[0].relative_to(ROOT))

print("Repository:", ROOT)
print("Python:", sys.executable)
print("GPU index:", ENV["GPU"])

Repository: /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton
Python: /home/hd/hd_hd/hd_nf184/cir_env/bin/python
GPU index: 0


### 1. Install dependencies

Install `requirements.txt` into the notebook's Python environment. Run this cell once in a new environment; if Jupyter requests a kernel restart, restart and rerun the setup cell.

In [2]:
run_command([sys.executable, "-m", "pip", "install", "-r", ROOT / "requirements.txt"])

Running: /home/hd/hd_hd/hd_nf184/cir_env/bin/python -m pip install -r /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/requirements.txt
Completed successfully.


### 2. Precompute model caches

`precompute.sh` runs the MLP decomposition, Attention OV decomposition, and residual precomputation code. It saves reusable files under `cache/`; run this step before the circuit experiments when those caches are missing.

In [11]:
run_command(["bash", ROOT / "precompute.sh"], environment=ENV)

Running: bash /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/precompute.sh
[START] GPT-2 MLP SVD cache
Device: cuda  device_map: None  torch_dtype: torch.float32
`torch_dtype` is deprecated! Use `dtype` instead!
Model: openai-community/gpt2  |  layers: 12  |  adapter: GPT2Adapter  |  device=cuda

Precomputing within-layer skeletons: 100%|██████████| 12/12 [00:00<00:00, 41323.19it/s]
Done.
[START] GPT-2 Attention OV cache
`torch_dtype` is deprecated! Use `dtype` instead!

Precomputing Attention W_OV: 100%|██████████| 12/12 [00:00<00:00, 35444.82it/s]
Attention OV cache ready: /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/cache/pre_compute_skeleton_attn_cache/openai-community/gpt2/attention_ov_cache
[START] GPT-2 residual circuit cache
[Cache] HIT base coupling/SVD: /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/cache/pre_compute_residual_cache/openai-community_gpt2/coupling/source_L4_targets_L5-L11.pt
`torch_dtype` is deprecated! Use `dtype` instead!
[Info] D

### 3. Static MLP circuit

`run_static.sh` calls `scripts_ske_cir/run_static_ske_per_neuron.py`. It evaluates the target token ` bird` on 100 prompts from the short dataset, comparing top, random, middle, and bottom path selections. The output reports mean target rank and probability for each condition.

In [8]:
run_command(["bash", ROOT / "run_static.sh"], environment=ENV,
            result_dir="results/static_ske_per_neuron_gpt2")

Running: bash /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/run_static.sh
[START] GPT-2 static: first 100 dataset prompts, target=" bird"
  [SKIP] exists: /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/results/static_ske_per_neuron_gpt2/openai-community_gpt2/short/bird_cd92815b_raw_dot_kpn2_L4-11_top-random-middle-bottom_pf3_rv6.json
[SKIP] all k_per_neuron values done for openai-community/gpt2
  model mean summary (complete) -> /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/results/mean_results/static_ske_per_neuron_gpt2/openai-community_gpt2/short/mean_summary.json
[RESULT] static k=2, prompts=100, target=' bird'
  Original: rank=1.03e+04, p=2.08e-05
  PerNeuron.top(gate): rank=1.11, p=0.0807
  PerNeuron.rnd(gate): rank=1.32e+03, p=0.000362
  PerNeuron.mid(gate): rank=2.87e+03, p=0.000131
  PerNeuron.bot(gate): rank=5.7e+03, p=4.49e-05
  JSON: /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/results/static_ske_per_neuron_gpt2/openai-community_gpt2/s

### 4. Dynamic MLP circuit

`run_dynm_mlp.sh` calls `scripts_ske_cir/run_skeleton_pred.py`. It analyzes the prompt `Janell Hobson analysed the imagery presented in the` with target ` report`, comparing selected and random paths under preservation and ablation. The output reports target rank and probability for each condition.

In [12]:
run_command(["bash", ROOT / "run_dynm_mlp.sh"], environment=ENV,
            result_dir="results/dynamic_ske_gpt2")

Running: bash /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/run_dynm_mlp.sh
[START] GPT-2 dynamic MLP prompt="Janell Hobson analysed the imagery presented in the" target=" report"
====== openai-community/gpt2 | target=' report' | layers 4-11 | 3/3 k values to run ======
Device: cuda  multi_gpu=False
`torch_dtype` is deprecated! Use `dtype` instead!

Target : ' report'  (id=989, decoded=' report', resolve='encode')
Layers : [4, 11]

Loading ske cache: 100%|██████████| 8/8 [00:00<00:00, 102.74layer/s]
  Precomputing delta/V/effector on GPU …

Precompute GPU tensors: 100%|██████████| 8/8 [00:00<00:00, 197.68layer/s]
  [GPU tensors ready for 8 layers]
  wrapped in_seq: 'Janell Hobson analysed the imagery presented in the'
  k_percent=0.01%  k1=236

k1=236:  75%|███████▌  | 3/4 [00:00<00:00,  7.17cond/s]
                                                       
  saved -> /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/results/dynamic_ske_gpt2/openai-community_gpt2/0_report_

### 5. Attention and Attention–MLP circuits

`run_dynm_attn.sh` calls `attn_mlp_cir.run_dynamic` for Attention alone and the combined Attention–MLP circuit. It uses the same prompt and target as the dynamic MLP example and writes separate JSON results for the two circuit types.

In [13]:
run_command(["bash", ROOT / "run_dynm_attn.sh"], environment=ENV,
            result_dir="results/attn_mlp_gpt2")

Running: bash /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/run_dynm_attn.sh
[START] GPT-2 dynamic type=attn_and_mlp
`torch_dtype` is deprecated! Use `dtype` instead!
[1/1] kp=1 -> /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/results/attn_mlp_gpt2/attn_and_mlp/openai-community_gpt2/inline_6e41fd006649/1_kp1p0_L4-11_attn_and_mlp_budget-global_ak-auto_norm-magnitude_preserving_rely-0_cond-ske_pre-ske_abl-rnd_pre-rnd_abl.json
[1/1] kp=5 -> /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/results/attn_mlp_gpt2/attn_and_mlp/openai-community_gpt2/inline_6e41fd006649/1_kp5p0_L4-11_attn_and_mlp_budget-global_ak-auto_norm-magnitude_preserving_rely-0_cond-ske_pre-ske_abl-rnd_pre-rnd_abl.json
[1/1] kp=10 -> /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/results/attn_mlp_gpt2/attn_and_mlp/openai-community_gpt2/inline_6e41fd006649/1_kp10p0_L4-11_attn_and_mlp_budget-global_ak-auto_norm-magnitude_preserving_rely-0_cond-ske_pre-ske_abl-rnd_pre-rnd_abl.json
[START] 

### 6. Residual circuit

`run_residual.sh` calls `circuit_skeleton.skeleton_analysis_residual.run`. It tests paths from a source neuron through residual subspaces to downstream neurons with zero and scale interventions. It prints aggregate comparisons and saves JSON results without generating plots.

In [15]:
run_command(["bash", ROOT / "run_residual.sh"], environment=ENV,
            result_dir="results/residual_gpt2")

Running: bash /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/run_residual.sh
[START] residual circuit intervention=zero
[Info] Resolving model dimensions for openai-community/gpt2 (start_layer=4) via meta-device inspection …
[Info] openai-community/gpt2: model_type=gpt2  n_layers=12  d_ff@L4=3072  total_params=0.16B
[Info] Resolved target_layers='5-11'  n_neurons=1
[Timer] Start: 2026-10-13 17:28:40
────────────────────────────────────────────────────────────
[Cache] HIT base coupling/SVD: /pfs/data6/home/hd/hd_hd/hd_nf184/NaNA_circuit_skeleton/cache/pre_compute_residual_cache/openai-community_gpt2/coupling/source_L4_targets_L5-L11.pt
[Info] d_ff=3072  neuron_selection=random  global_n_neurons=1  shard=0/1  shard_n_neurons=1
`torch_dtype` is deprecated! Use `dtype` instead!
[Info] Dataset overlap validation passed: unique IDs and prompts within each split; no ID or prompt overlap across splits.
[Info] Dataset split: test=[0, 100) coefficient=[1000, 2000)
Resume[fc78134dd8c8]: 